In [21]:
import pandas as pd
import json
import os

print("🔍 BẮT ĐẦU CHỤP X-QUANG DỮ LIỆU THÔ (RAW DATA PROFILING)...\n")
base_path = "../data/1_bronze"

# =========================================================================
# 1. NẠP DỮ LIỆU THÔ ĐỂ DÒ LỖI (KHÔNG LÀM SẠCH)
# =========================================================================
# Chỉ đọc 100k dòng transactions để khảo sát nhanh, tránh treo RAM
df_trans_raw = pd.read_csv(os.path.join(base_path, "transactions_data.csv"), nrows=100000)
df_users_raw = pd.read_csv(os.path.join(base_path, "users_data.csv"))
df_cards_raw = pd.read_csv(os.path.join(base_path, "cards_data.csv"))

with open(os.path.join(base_path, "mcc_codes.json"), "r", encoding="utf-8") as f:
    mcc_dict = json.load(f)
df_mcc_raw = pd.DataFrame(list(mcc_dict.items()), columns=['mcc', 'merchant_category'])

with open(os.path.join(base_path, "train_fraud_labels.json"), "r", encoding="utf-8") as f:
    fraud_dict = json.load(f)
df_labels_raw = pd.DataFrame(list(fraud_dict.items()), columns=['transaction_id', 'is_fraud'])

datasets = {
    "Bảng Transactions (Thô)": df_trans_raw,
    "Bảng Users Profiles (Thô)": df_users_raw,
    "Bảng Cards Profiles (Thô)": df_cards_raw,
    "Bảng Danh mục MCC (Thô)": df_mcc_raw,
    "Bảng Nhãn Fraud (Thô)": df_labels_raw
}

# =========================================================================
# 2. MÁY QUÉT TỰ ĐỘNG: TÌM LỖ KHUYẾT THIẾU & TRÙNG LẶP
# =========================================================================
for name, df in datasets.items():
    print("="*60)
    print(f"📊 BÁO CÁO DÒ LỖI: {name.upper()}")
    print("="*60)
    
    print(f"🔹 Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
    
    # Dò lỗi Khuyết thiếu (Missing Values)
    missing_data = df.isna().sum()
    print("\n⚠️ Lỗi Khuyết thiếu (NaN):")
    if missing_data.sum() > 0:
        missing_df = pd.DataFrame({'Số dòng trống': missing_data, 'Tỷ lệ (%)': (missing_data / len(df)) * 100})
        print(missing_df[missing_df['Số dòng trống'] > 0])
    else:
        print("👉 Sạch sẽ: Không có cột nào bị rỗng.")
        
    # Dò lỗi Trùng lặp (Duplicates)
    print("\n⚠️ Lỗi Trùng lặp (Duplicates):")
    try:
        dup_count = df.duplicated().sum()
        print(f"👉 Số dòng bị trùng lặp hoàn toàn: {dup_count}")
    except TypeError:
        print("👉 (Bỏ qua do chứa cấu trúc phức tạp unhashable)")
        
    # In ra kiểu dữ liệu để phát hiện lỗi định dạng (VD: số nhưng bị nhận là chữ)
    print("\n⚠️ Kiểu dữ liệu (Dtypes) đang bị nhận diện:")
    print(df.dtypes)
    print("\n" + "-"*60 + "\n")

# =========================================================================
# 3. KÍNH HIỂN VI: SOI SÂU VÀO CÁC CỘT DỄ DÍNH LỖI (DEEP DIVE)
# =========================================================================
print("🎯 BẮT ĐẦU SOI SÂU VÀO TỪNG CỘT CỤ THỂ...\n")

print("1. Tỷ lệ mất cân bằng nhãn AI (Fraud Labels):")
print(df_labels_raw['is_fraud'].value_counts(dropna=False, normalize=True) * 100)
print("-" * 60)

print("2. Lỗi định dạng rác trong phương thức quẹt thẻ (use_chip):")
print(df_trans_raw['use_chip'].value_counts(dropna=False))
print("-" * 60)

print("3. Tỷ lệ trống và các mã lỗi thực tế của hệ thống (errors):")
print(df_trans_raw['errors'].value_counts(dropna=False))
print("-" * 60)

print("4. Soi cấu trúc văn bản của cột số dư/thu nhập (để xem có dính dấu $ hoặc , không):")
print("Bảng Transactions - Cột 'amount' (3 dòng đầu):")
print(df_trans_raw['amount'].head(3).tolist())
print("\nBảng Users - Cột 'yearly_income' (3 dòng đầu):")
print(df_users_raw['yearly_income'].head(3).tolist())

🔍 BẮT ĐẦU CHỤP X-QUANG DỮ LIỆU THÔ (RAW DATA PROFILING)...

📊 BÁO CÁO DÒ LỖI: BẢNG TRANSACTIONS (THÔ)
🔹 Kích thước: 100000 dòng, 12 cột

⚠️ Lỗi Khuyết thiếu (NaN):
                Số dòng trống  Tỷ lệ (%)
merchant_state          11016     11.016
zip                     11484     11.484
errors                  98448     98.448

⚠️ Lỗi Trùng lặp (Duplicates):
👉 Số dòng bị trùng lặp hoàn toàn: 0

⚠️ Kiểu dữ liệu (Dtypes) đang bị nhận diện:
id                  int64
date                  str
client_id           int64
card_id             int64
amount                str
use_chip              str
merchant_id         int64
merchant_city         str
merchant_state        str
zip               float64
mcc                 int64
errors                str
dtype: object

------------------------------------------------------------

📊 BÁO CÁO DÒ LỖI: BẢNG USERS PROFILES (THÔ)
🔹 Kích thước: 2000 dòng, 14 cột

⚠️ Lỗi Khuyết thiếu (NaN):
👉 Sạch sẽ: Không có cột nào bị rỗng.

⚠️ Lỗi Trùng lặp (Duplicates

In [22]:
# =========================================================================
# 📊 CELL BỔ SUNG: DÒ NGƯỠNG OUTLIER CHUYÊN SÂU (DEEP-DIVE OUTLIERS)
# =========================================================================
print("🎯 BẮT ĐẦU SĂN TÌM GIÁ TRỊ NGOẠI LAI (OUTLIERS)...\n")

# 1. Khảo sát cột 'amount' ở bảng Transactions (Sample)
print("1. BẢNG TRANSACTIONS - CỘT 'AMOUNT'")
print("-" * 40)
# Tạm thời dọn sạch ký tự để ép kiểu tính toán
trans_amount_clean = df_trans_raw['amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.replace(r'^\((.+)\)$', r'-\1', regex=True).astype(float)

print("🔹 Thống kê mô tả tổng quan:")
print(trans_amount_clean.describe())
print("\n🔹 Chi tiết các mốc phân vị cao (Phát hiện điểm nhảy vọt):")
print(trans_amount_clean.quantile([0.90, 0.95, 0.98, 0.99, 0.995, 0.999]))
print("=" * 60 + "\n")


# 2. Khảo sát các cột tiền tệ ở bảng Users Profiles
print("2. BẢNG USERS PROFILES - CÁC CỘT THU NHẬP & NỢ")
print("-" * 40)
user_cols = ['per_capita_income', 'yearly_income', 'total_debt']
for col in user_cols:
    if col in df_users_raw.columns:
        user_col_clean = df_users_raw[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.replace(r'^\((.+)\)$', r'-\1', regex=True).astype(float)
        print(f"🔹 Cột '{col}' - Ngưỡng cực đại (Max): {user_col_clean.max()}")
        print(f"🔹 Cột '{col}' - Mốc phân vị 99% (Ngưỡng thông thường): {user_col_clean.quantile(0.99)}")
        print(f"🔹 Cột '{col}' - Mốc phân vị 99.9%: {user_col_clean.quantile(0.999)}\n")
print("=" * 60 + "\n")


# 3. Khảo sát cột 'credit_limit' ở bảng Cards Profiles
print("3. BẢNG CARDS PROFILES - CỘT 'CREDIT_LIMIT'")
print("-" * 40)
if 'credit_limit' in df_cards_raw.columns:
    card_limit_clean = df_cards_raw['credit_limit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.replace(r'^\((.+)\)$', r'-\1', regex=True).astype(float)
    print("🔹 Thống kê mô tả tổng quan:")
    print(card_limit_clean.describe())
    print("\n🔹 Mốc phân vị cao:")
    print(card_limit_clean.quantile([0.95, 0.99, 0.999]))

🎯 BẮT ĐẦU SĂN TÌM GIÁ TRỊ NGOẠI LAI (OUTLIERS)...

1. BẢNG TRANSACTIONS - CỘT 'AMOUNT'
----------------------------------------
🔹 Thống kê mô tả tổng quan:
count    100000.000000
mean         43.120798
std          82.952348
min        -500.000000
25%           8.860000
50%          28.940000
75%          64.560000
max        2807.050000
Name: amount, dtype: float64

🔹 Chi tiết các mốc phân vị cao (Phát hiện điểm nhảy vọt):
0.900    106.17200
0.950    145.55150
0.980    219.37040
0.990    318.00000
0.995    451.04370
0.999    874.20773
Name: amount, dtype: float64

2. BẢNG USERS PROFILES - CÁC CỘT THU NHẬP & NỢ
----------------------------------------
🔹 Cột 'per_capita_income' - Ngưỡng cực đại (Max): 163145.0
🔹 Cột 'per_capita_income' - Mốc phân vị 99% (Ngưỡng thông thường): 58537.75999999998
🔹 Cột 'per_capita_income' - Mốc phân vị 99.9%: 137441.15499999968

🔹 Cột 'yearly_income' - Ngưỡng cực đại (Max): 307018.0
🔹 Cột 'yearly_income' - Mốc phân vị 99% (Ngưỡng thông thường): 118866.4599

In [23]:
#Import hàm làm sạch
import sys
sys.path.append('..')  # Vì notebook nằm trong /notebooks/, cần lùi 1 cấp lên src/

from src.etl.transform import clean_users_data, clean_cards_data, clean_transaction_chunk

print("✅ Import thành công")

✅ Import thành công


In [24]:

# Chạy các hàm làm sạch trên data thô đã đọc ở trên
df_users_clean = clean_users_data(df_users_raw.copy())
df_cards_clean = clean_cards_data(df_cards_raw.copy())
df_tx_clean    = clean_transaction_chunk(df_trans_raw.copy())

print("✅ Đã làm sạch xong 3 bảng")

2026-06-13 17:37:19,731 - INFO - Đang làm sạch dữ liệu Users...
2026-06-13 17:37:19,770 - INFO - Đang làm sạch dữ liệu Cards...


✅ Đã làm sạch xong 3 bảng


In [25]:
import importlib
import src.etl.transform as t
importlib.reload(t)
from src.etl.transform import clean_users_data, clean_cards_data, clean_transaction_chunk

df_users_clean = clean_users_data(df_users_raw.copy())
df_cards_clean = clean_cards_data(df_cards_raw.copy())
df_tx_clean    = clean_transaction_chunk(df_trans_raw.copy())

print("✅ Reload và làm sạch xong")
print("Kiểu amount:", df_tx_clean['amount'].dtype)  # Phải là float64

2026-06-13 17:37:20,398 - INFO - Đang làm sạch dữ liệu Users...
2026-06-13 17:37:20,436 - INFO - Đang làm sạch dữ liệu Cards...


✅ Reload và làm sạch xong
Kiểu amount: float64


In [26]:
print("=" * 50)
print("KIỂM TRA USERS - 3 cột tiền tệ")
print("=" * 50)
print(df_users_clean[['per_capita_income','yearly_income','total_debt']].dtypes)
# Kỳ vọng: float64, không phải object

print("\nGiá trị mẫu (không được có $):")
print(df_users_clean[['per_capita_income','yearly_income','total_debt']].head(3))

print("\n" + "=" * 50)
print("KIỂM TRA CARDS - credit_limit và card_type")
print("=" * 50)
print(df_cards_clean[['credit_limit','card_type']].dtypes)
print(df_cards_clean[['credit_limit','card_type']].head(3))
print("\ncard_type unique:", df_cards_clean['card_type'].unique())
# Kỳ vọng: toàn chữ thường

print("\n" + "=" * 50)
print("KIỂM TRA TRANSACTIONS - amount, outlier, datetime")
print("=" * 50)
print("Kiểu dữ liệu amount:", df_tx_clean['amount'].dtype)
print("Max amount sau clip:", df_tx_clean['amount'].max())
print("Có giá trị âm (Refund)?", (df_tx_clean['amount'] < 0).any())
print("\nCác cột datetime mới:")
print("Các cột trong df_tx_clean:", df_tx_clean.columns.tolist())
print(df_tx_clean[['tx_hour','tx_day_of_week','is_night_tx']].head(5))

KIỂM TRA USERS - 3 cột tiền tệ
per_capita_income    float64
yearly_income        float64
total_debt           float64
dtype: object

Giá trị mẫu (không được có $):
   per_capita_income  yearly_income  total_debt
0            29278.0        59696.0    127613.0
1            37891.0        77254.0    191349.0
2            22681.0        33483.0       196.0

KIỂM TRA CARDS - credit_limit và card_type
credit_limit    float64
card_type           str
dtype: object
   credit_limit card_type
0       24295.0     debit
1       21968.0     debit
2       46414.0     debit

card_type unique: <ArrowStringArray>
['debit', 'credit', 'debit (prepaid)']
Length: 3, dtype: str

KIỂM TRA TRANSACTIONS - amount, outlier, datetime
Kiểu dữ liệu amount: float64
Max amount sau clip: 318.0
Có giá trị âm (Refund)? True

Các cột datetime mới:
Các cột trong df_tx_clean: ['transaction_id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'tx_

In [28]:
p99_raw = df_trans_raw['amount'].str.replace('$','',regex=False).str.replace(',','',regex=False).str.replace(r'^\((.+)\)$', r'-\1', regex=True).astype(float).quantile(0.99)

checks = {
    'per_capita_income là float' : 'float' in str(df_users_clean['per_capita_income'].dtype),
    'yearly_income là float'     : 'float' in str(df_users_clean['yearly_income'].dtype),
    'total_debt là float'        : 'float' in str(df_users_clean['total_debt'].dtype),
    'credit_limit là float'      : 'float' in str(df_cards_clean['credit_limit'].dtype),
    'card_type toàn chữ thường'  : not any(v != v.lower() for v in df_cards_clean['card_type'].unique() if isinstance(v,str)),
    'amount là float'            : 'float' in str(df_tx_clean['amount'].dtype),
    'amount không còn $'         : not df_tx_clean['amount'].astype(str).str.contains(r'\$').any(),
    'Outlier đã clip tại p99'    : df_tx_clean['amount'].max() <= p99_raw + 0.01,
    'Có cột tx_hour'             : 'tx_hour' in df_tx_clean.columns,
    'Có cột tx_day_of_week'      : 'tx_day_of_week' in df_tx_clean.columns,
    'Có cột is_night_tx'         : 'is_night_tx' in df_tx_clean.columns,
}

passed = sum(checks.values())
total  = len(checks)

print("=" * 50)
print("       BẢNG TỔNG KẾT KIỂM TRA")
print("=" * 50)
for name, ok in checks.items():
    print(f'{"✅ PASS" if ok else "❌ FAIL"}  |  {name}')

print(f"\n🎯 KẾT QUẢ: {passed}/{total} tiêu chí đạt")
if passed == total:
    print("🎉 Code làm sạch chạy CHUẨN!")
else:
    print("⚠️  Còn lỗi — kiểm tra lại các dòng ❌")

       BẢNG TỔNG KẾT KIỂM TRA
✅ PASS  |  per_capita_income là float
✅ PASS  |  yearly_income là float
✅ PASS  |  total_debt là float
✅ PASS  |  credit_limit là float
✅ PASS  |  card_type toàn chữ thường
✅ PASS  |  amount là float
✅ PASS  |  amount không còn $
✅ PASS  |  Outlier đã clip tại p99
✅ PASS  |  Có cột tx_hour
✅ PASS  |  Có cột tx_day_of_week
✅ PASS  |  Có cột is_night_tx

🎯 KẾT QUẢ: 11/11 tiêu chí đạt
🎉 Code làm sạch chạy CHUẨN!
